# ComicAnalizer - OCR Only Desde Magi Ya Procesado

Usa este notebook cuando ya tienes un ZIP con resultados Magi de Colab y no quieres volver a procesar Magi. Sube:

- `magi_clean_full.zip`: dataset limpio con imagenes.
- `colab_clean_full_magi_ocr_outputs.zip`: ZIP anterior que contiene `outputs/runs/colab_clean_full/magi`.

El notebook ejecuta PaddleOCR, genera overlays OCR, evidencia OCR y descarga un ZIP nuevo.


In [ ]:
!rm -rf /content/ComicAnalizer
!git clone https://github.com/nicolas4432/ComicAnalizer.git /content/ComicAnalizer
%cd /content/ComicAnalizer
!git log --oneline -5


In [ ]:
%cd /content/ComicAnalizer

import subprocess, sys

commands = [
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'setuptools',
        'shapely',
    ],
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'paddlepaddle==3.1.1',
        '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cpu/',
    ],
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'paddleocr==3.5.0',
    ],
]

for cmd in commands:
    print('Instalando:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

print('Verificando PaddleOCR...')
import paddle
import paddleocr
print('paddle:', paddle.__version__)
print('paddleocr:', getattr(paddleocr, '__version__', 'unknown'))
print('paddle cuda disponible:', paddle.is_compiled_with_cuda())


## Subir Entradas

Sube los dos ZIPs. El ZIP de Magi puede ser el que acabas de descargar aunque OCR haya fallado; la parte Magi sigue siendo util.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
print('Subidos:', list(uploaded))

DATASET_ZIP = next(name for name in uploaded if name.startswith('magi_') and 'clean' in name and name.endswith('.zip'))
MAGI_RESULTS_ZIP = next(name for name in uploaded if name != DATASET_ZIP and name.endswith('.zip'))
PACKAGE_NAME = DATASET_ZIP.replace('.zip', '')

!rm -rf /content/magi_sample /content/magi_results_input
!mkdir -p /content/magi_sample /content/magi_results_input
!unzip -q -o "$DATASET_ZIP" -d /content/magi_sample
!unzip -q -o "$MAGI_RESULTS_ZIP" -d /content/magi_results_input

print('DATASET_ZIP:', DATASET_ZIP)
print('PACKAGE_NAME:', PACKAGE_NAME)
print('MAGI_RESULTS_ZIP:', MAGI_RESULTS_ZIP)
!find /content/magi_sample -maxdepth 5 -type d | head -30
!find /content/magi_results_input -maxdepth 7 -name magi_results.json -o -name metrics.json | head -20


## Configurar OCR

Por defecto corre OCR en todas las paginas (`OCR_LIMIT = 0`). Si ves que va muy lento, det?n la celda: el script va guardando `paddle_magi_ocr_comparison.partial.json` con lo ya procesado.


In [ ]:
from pathlib import Path
import shlex
import subprocess

RUN_NAME = 'colab_clean_full_ocr_full'
SOURCE_RUN_NAME = 'colab_clean_full'
DATASET_NAME = 'test_1_clean'
COMIC_ID = ''  # '' = todos; por ejemplo 'nekkorarekko' para filtrar
OCR_LIMIT = 0  # 0 = todas las paginas seleccionadas
OCR_SELECTION = 'first'  # first conserva el orden de Magi; random/suspicious tambien existen

RUN_ROOT = f'outputs/runs/{RUN_NAME}'
IMAGE_ROOT = f'/content/magi_sample/{PACKAGE_NAME}/by_comic'
MAGI_INPUT = f'/content/magi_results_input/outputs/runs/{SOURCE_RUN_NAME}/magi'
ANALYSIS_OUTPUT = f'{RUN_ROOT}/analysis/magi_analysis_report.json'
OCR_OUTPUT = f'{RUN_ROOT}/analysis/paddle_magi_ocr_comparison.json'
OCR_VISUALS = f'{RUN_ROOT}/visuals/ocr_boxes'
OCR_EVIDENCE_OUTPUT = f'{RUN_ROOT}/analysis/ocr_evidence'

print('MAGI_INPUT existe:', Path(MAGI_INPUT).exists(), MAGI_INPUT)
print('IMAGE_ROOT existe:', Path(IMAGE_ROOT).exists(), IMAGE_ROOT)


In [ ]:
!python -m tools.analyze_magi_results \
  --input "$MAGI_INPUT" \
  --output "$ANALYSIS_OUTPUT" \
  --top-n 30


In [ ]:
cmd = [
    'python', '-m', 'tools.compare_magi_paddleocr',
    '--magi-input', MAGI_INPUT,
    '--image-root', IMAGE_ROOT,
    '--dataset-name', DATASET_NAME,
    '--selection', OCR_SELECTION,
    '--limit', str(OCR_LIMIT),
    '--seed', '42',
    '--lang', 'en',
    '--visual-output-dir', OCR_VISUALS,
    '--output', OCR_OUTPUT,
    '--checkpoint-every', '1',
]
if COMIC_ID:
    cmd.extend(['--comic-id', COMIC_ID])

print('Ejecutando OCR:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
from pathlib import Path

ocr_path = Path(OCR_OUTPUT)
partial_path = ocr_path.with_suffix('.partial.json')
read_path = ocr_path if ocr_path.exists() else partial_path
ocr_report = json.loads(read_path.read_text())
print('Leyendo:', read_path)
print(json.dumps(ocr_report['summary'], indent=2, ensure_ascii=False))
for item in ocr_report['comparisons'][:20]:
    print(item['comic_id'], item['file_name'], 'Magi=', item['magi_text_regions'], 'Paddle=', item['paddle_text_blocks'], 'match=', item['matched_regions'], 't=', round(item['paddle_elapsed_seconds'], 2), 'err=', item['paddle_error'])


In [ ]:
if Path(OCR_OUTPUT).exists():
    cmd = [
        'python', '-m', 'tools.export_ocr_evidence',
        '--ocr-report', OCR_OUTPUT,
        '--magi-input', MAGI_INPUT,
        '--output-dir', OCR_EVIDENCE_OUTPUT,
    ]
    print('Exportando evidencia OCR:')
    print(' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, check=True)
else:
    print('OCR final no existe; si detuviste la corrida, descarga el partial igualmente.')


## Descargar Resultados OCR

Descarga el run OCR, incluyendo JSON final o parcial, overlays y evidencia si se genero.


In [ ]:
from google.colab import files

zip_out = f'{RUN_NAME}_outputs.zip'
!zip -qr "$zip_out" "$RUN_ROOT"
files.download(zip_out)
